# Génération et Sauvegarde des Embeddings Uni-Mol

Ce notebook permet de calculer les représentations (embeddings) des ~5000 molécules du dataset d'odeurs (`data/smiles_odor_dataset.csv`) avec le modèle 3D **Uni-Mol** (`unimol-tools`), puis de les enregistrer une bonne fois pour toutes dans un fichier PyTorch (`data/unimol_embeddings.pt`).

### Contenu du fichier sauvegardé :
- **`embeddings`** : Tenseur PyTorch `torch.FloatTensor` de dimension `(N, 512)` représentant chaque molécule dans l'espace latent Uni-Mol.
- **`labels`** : Tenseur PyTorch `torch.FloatTensor` de dimension `(N, 138)` contenant les indicateurs binaires des 138 descripteurs d'odeur.
- **`smiles`** : Liste Python des chaînes SMILES des `N` molécules.
- **`label_names`** : Noms des 138 descripteurs olfactifs.
- **`model_name`** : Nom du modèle utilisé (`unimolv1`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from unimol_tools import UniMolRepr

# Résolution du chemin racine du projet
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "scripts":
    ROOT_DIR = ROOT_DIR.parent.parent

CSV_PATH = ROOT_DIR / "data" / "smiles_odor_dataset.csv"
OUTPUT_DIR = ROOT_DIR / "data"
OUTPUT_FILE = OUTPUT_DIR / "unimol_embeddings.pt"

BATCH_SIZE = 64
MODEL_NAME = "unimolv1"
USE_CUDA = torch.cuda.is_available()

print(f"Racine du projet   : {ROOT_DIR}")
print(f"Fichier CSV source : {CSV_PATH}")
print(f"Fichier de sortie  : {OUTPUT_FILE}")
print(f"CUDA disponible    : {USE_CUDA}")

c:\_Donnees\Polytech\INFO5\PRED\ml-odomol-classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Racine du projet   : C:\_Donnees\Polytech\INFO5\PRED\ml-odomol-classifier
Fichier CSV source : C:\_Donnees\Polytech\INFO5\PRED\ml-odomol-classifier\data\smiles_odor_dataset.csv
Fichier de sortie  : C:\_Donnees\Polytech\INFO5\PRED\ml-odomol-classifier\data\unimol_embeddings.pt
CUDA disponible    : False


## 1. Chargement et inspection du Dataset

In [2]:
assert CSV_PATH.exists(), f"Le fichier {CSV_PATH} est introuvable !"
df = pd.read_csv(CSV_PATH)

smiles_col = df.columns[0]  # "nonStereoSMILES"
descriptor_col = df.columns[1]  # "descriptors"
label_cols = list(df.columns[2:])  # 138 classes d'odeurs

smiles_list = df[smiles_col].astype(str).tolist()
labels_matrix = df[label_cols].to_numpy(dtype=np.float32)

print(f"Nombre total de molécules : {len(smiles_list)}")
print(f"Nombre de descripteurs olfactifs : {len(label_cols)}")
df.head(3)

Nombre total de molécules : 4983
Nombre de descripteurs olfactifs : 138


,nonStereoSMILES,descriptors,alcoholic,aldehydic,alliaceous,almond,amber,animal,anisic,apple,...,tropical,vanilla,vegetable,vetiver,violet,warm,waxy,weedy,winey,woody
0,CC(O)CN,fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CCC(=O)C(=O)O,fatty;lactonic;sweet;caramellic;creamy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,O=C(O)CCc1ccccc1,rose;floral;fatty;sweet;musk;cinnamon;balsamic,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. Initialisation du modèle Uni-Mol

In [4]:
print(f"Initialisation de UniMolRepr ({MODEL_NAME})...")
clf = UniMolRepr(
    data_type="molecule",
    remove_hs=False,
    model_name=MODEL_NAME,
    use_cuda=USE_CUDA
)
print("Modèle Uni-Mol prêt.")

2026-10-14 12:07:40 | unimol_tools\models\unimol.py | 169 | INFO | Uni-Mol Tools | Loading pretrained weights from c:\_Donnees\Polytech\INFO5\PRED\ml-odomol-classifier\.venv\Lib\site-packages\unimol_tools\weights\mol_pre_all_h_220816.pt


Initialisation de UniMolRepr (unimolv1)...
Modèle Uni-Mol prêt.


## 3. Génération des Embeddings par lots

Pour chaque molécule, Uni-Mol calcule les coordonnées 3D (conformères RDKit) puis applique son modèle Transformer pré-entraîné pour produire un vecteur CLS de dimension 512.

In [5]:
all_embeddings = []
valid_indices = []
failed_molecules = []

total_mols = len(smiles_list)
batches = [
    (i, smiles_list[i : i + BATCH_SIZE], list(range(i, min(i + BATCH_SIZE, total_mols))))
    for i in range(0, total_mols, BATCH_SIZE)
]

for start_idx, batch_smiles, batch_indices in tqdm(batches, desc="Calcul Uni-Mol"):
    try:
        reprs = clf.get_repr(batch_smiles, return_atomic_reprs=False)
        batch_emb = np.stack(reprs, axis=0)
        all_embeddings.append(batch_emb)
        valid_indices.extend(batch_indices)
    except Exception as e:
        print(f"Avertissement lot {start_idx} : traitement individuel ({e})")
        for idx, s in zip(batch_indices, batch_smiles):
            try:
                single_repr = clf.get_repr([s], return_atomic_reprs=False)
                all_embeddings.append(np.array(single_repr[0]).reshape(1, -1))
                valid_indices.append(idx)
            except Exception as single_err:
                failed_molecules.append((idx, s, str(single_err)))

# Assemblage sous forme de tenseurs PyTorch
embeddings_np = np.concatenate(all_embeddings, axis=0)
embeddings_tensor = torch.from_numpy(embeddings_np).float()
labels_tensor = torch.from_numpy(labels_matrix[valid_indices]).float()
filtered_smiles = [smiles_list[idx] for idx in valid_indices]

print(f"Forme des embeddings : {embeddings_tensor.shape}")
print(f"Forme des labels     : {labels_tensor.shape}")
print(f"Molécules traitées   : {len(filtered_smiles)} / {total_mols}")
if failed_molecules:
    print(f"Molécules en échec  : {len(failed_molecules)}")

100%|██████████| 64/64 [00:00<00:00, 199.49it/s]
2026-10-14 12:07:44 | unimol_tools\data\conformer.py | 199 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2026-10-14 12:07:44 | unimol_tools\data\conformer.py | 216 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 100.00% of molecules.
2026-10-14 12:07:44 | unimol_tools\tasks\trainer.py | 103 | INFO | Uni-Mol Tools | Using CPU.
100%|██████████| 64/64 [00:00<00:00, 90.15it/s]
2026-10-14 12:07:46 | unimol_tools\data\conformer.py | 199 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2026-10-14 12:07:46 | unimol_tools\data\conformer.py | 216 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 100.00% of molecules.
2026-10-14 12:07:46 | unimol_tools\tasks\trainer.py | 103 | INFO | Uni-Mol Tools | Using CPU.
100%|██████████| 64/64 [00:00<00:00, 121.45it/s]
2026-10-14 12:07:49 | unimol_tools\data\conformer.py | 199 | INFO | Uni-Mol Tool

Forme des embeddings : torch.Size([4983, 512])
Forme des labels     : torch.Size([4983, 138])
Molécules traitées   : 4983 / 4983


## 4. Sauvegarde dans un fichier PyTorch (.pt)

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

payload = {
    "embeddings": embeddings_tensor,
    "labels": labels_tensor,
    "smiles": filtered_smiles,
    "label_names": label_cols,
    "model_name": MODEL_NAME,
}

torch.save(payload, OUTPUT_FILE)
size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f"Sauvegarde réussie dans : {OUTPUT_FILE}")
print(f"Taille du fichier : {size_mb:.2f} Mo")

Sauvegarde réussie dans : C:\_Donnees\Polytech\INFO5\PRED\ml-odomol-classifier\data\unimol_embeddings.pt
Taille du fichier : 12.50 Mo


## 5. Vérification du fichier sauvegardé

In [7]:
loaded = torch.load(OUTPUT_FILE, map_location="cpu", weights_only=False)

print("Clés enregistrées :", list(loaded.keys()))
print(f"Embeddings shape   : {loaded['embeddings'].shape} (dtype: {loaded['embeddings'].dtype})")
print(f"Labels shape       : {loaded['labels'].shape} (dtype: {loaded['labels'].dtype})")
print(f"Nombre de SMILES   : {len(loaded['smiles'])}")
print(f"Nombre de classes  : {len(loaded['label_names'])}")
print(f"Présence de NaN    : {torch.isnan(loaded['embeddings']).any().item()}")
print(f"Premier SMILES     : {loaded['smiles'][0]}")
print(f"Extrait embedding  : {loaded['embeddings'][0, :5].tolist()}")

Clés enregistrées : ['embeddings', 'labels', 'smiles', 'label_names', 'model_name']
Embeddings shape   : torch.Size([4983, 512]) (dtype: torch.float32)
Labels shape       : torch.Size([4983, 138]) (dtype: torch.float32)
Nombre de SMILES   : 4983
Nombre de classes  : 138
Présence de NaN    : False
Premier SMILES     : CC(O)CN
Extrait embedding  : [0.24528008699417114, -1.5320850610733032, 1.963990569114685, -0.9636042714118958, -0.6590214371681213]
